<a href="https://colab.research.google.com/github/Purposefulpaul/Miva_Ada_Siwes_team/blob/MIva_Labworks/neural_net_labwork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1: Baseline Model Construction

1.1 Setup & Data Loading

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import time

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Data transforms with augmentation for better generalization
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10 dataset
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

# Create data loaders
def get_dataloaders(batch_size=32):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, test_loader

# Class names for CIFAR-10
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

Using device: cuda


100%|██████████| 170M/170M [48:35<00:00, 58.5kB/s]


1.2 Baseline CNN Architecture

In [3]:
class BaselineCNN(nn.Module):
    """
    Baseline CNN with:
    - 2 Convolutional blocks (Conv -> ReLU -> MaxPool)
    - 1 Hidden FC layer (128 units)
    - Output layer (10 units with Softmax)
    """
    def __init__(self, activation='relu', num_blocks=2):
        super(BaselineCNN, self).__init__()

        # Activation functions
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'tanh':
            self.activation = nn.Tanh()
        elif activation == 'leaky_relu':
            self.activation = nn.LeakyReLU(0.01)
        else:
            self.activation = nn.ReLU()

        # Convolutional blocks
        self.conv_layers = nn.ModuleList()
        in_channels = 3
        filters = [32, 64, 128]  # Filters for each block

        for i in range(num_blocks):
            self.conv_layers.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, filters[i], kernel_size=3, padding=1),
                    nn.BatchNorm2d(filters[i]),  # Added for stability
                    self.activation,
                    nn.MaxPool2d(2, 2)
                )
            )
            in_channels = filters[i]

        # Calculate flattened size dynamically
        # Pass a dummy tensor through the convolutional layers to determine the size
        dummy_input = torch.zeros(1, 3, 32, 32) # Batch_size 1, 3 channels, 32x32 image
        with torch.no_grad():
            x = dummy_input
            for conv_block in self.conv_layers:
                x = conv_block(x)
            self.flattened_size = x.view(x.size(0), -1).size(1)

        # Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            self.activation,
            nn.Dropout(0.5),  # Prevent overfitting
            nn.Linear(128, 10)
        )

    def forward(self, x):
        for conv_block in self.conv_layers:
            x = conv_block(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc_layers(x)
        return x  # No Softmax here; we'll use CrossEntropyLoss

1.3 Training & Evaluation Functions

In [4]:
def train_model(model, train_loader, test_loader, optimizer, criterion, epochs=15):
    """
    Train a model and track metrics for analysis.
    """
    model = model.to(device)
    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': []
    }

    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total

        # Evaluation phase
        test_loss, test_acc = evaluate_model(model, test_loader, criterion)

        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)

        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, '
              f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%')

    return history

def evaluate_model(model, test_loader, criterion):
    """Evaluate model on test set."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_loss = running_loss / len(test_loader)
    test_acc = 100. * correct / total
    return test_loss, test_acc

1.4 Run Baseline

In [5]:
def run_experiment(config_name, model_config, optimizer_config, batch_size, epochs=15):
    """Run a single experiment with given configuration."""
    print(f"\n{'='*60}")
    print(f"Running Experiment: {config_name}")
    print(f"{'='*60}")

    # Get data loaders with specified batch size
    train_loader, test_loader = get_dataloaders(batch_size)

    # Create model
    model = BaselineCNN(
        activation=model_config.get('activation', 'relu'),
        num_blocks=model_config.get('num_blocks', 2)
    )

    # Create optimizer
    if optimizer_config['type'] == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=optimizer_config.get('lr', 0.001))
    elif optimizer_config['type'] == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=optimizer_config.get('lr', 0.01),
                              momentum=0.9)
    elif optimizer_config['type'] == 'rmsprop':
        optimizer = optim.RMSprop(model.parameters(), lr=optimizer_config.get('lr', 0.001))

    criterion = nn.CrossEntropyLoss()

    start_time = time.time()
    history = train_model(model, train_loader, test_loader, optimizer, criterion, epochs)
    end_time = time.time()

    return {
        'config': config_name,
        'history': history,
        'final_test_acc': history['test_acc'][-1],
        'training_time': end_time - start_time,
        'model': model
    }

# Run Baseline
baseline_config = {
    'activation': 'relu',
    'num_blocks': 2
}
optimizer_config = {'type': 'adam', 'lr': 0.001}

baseline_results = run_experiment(
    'Baseline (2 blocks, ReLU, Adam, bs=32)',
    baseline_config,
    optimizer_config,
    batch_size=32,
    epochs=15
)

print(f"\nBaseline Test Accuracy: {baseline_results['final_test_acc']:.2f}%")


Running Experiment: Baseline (2 blocks, ReLU, Adam, bs=32)


Epoch 1/15: 100%|██████████| 1563/1563 [00:25<00:00, 62.15it/s]


Epoch 1: Train Loss: 1.8748, Train Acc: 28.17%, Test Loss: 1.6143, Test Acc: 39.05%


Epoch 2/15: 100%|██████████| 1563/1563 [00:28<00:00, 55.01it/s]


Epoch 2: Train Loss: 1.7003, Train Acc: 35.01%, Test Loss: 1.3729, Test Acc: 51.23%


...

Epoch 13: Train Loss: 1.3666, Train Acc: 49.02%, Test Loss: 0.9904, Test Acc: 65.94%


Epoch 14/15: 100%|██████████| 1563/1563 [00:25<00:00, 61.55it/s]


Epoch 14: Train Loss: 1.3366, Train Acc: 50.29%, Test Loss: 0.9767, Test Acc: 66.71%


Epoch 15/15: 100%|██████████| 1563/1563 [00:25<00:00, 62.26it/s]


Epoch 15: Train Loss: 1.3094, Train Acc: 51.58%, Test Loss: 0.9464, Test Acc: 67.48%

Baseline Test Accuracy: 67.48%


Part 2: Controlled Experiments

Experiment A: Number of Hidden Layers

In [6]:
def experiment_A():
    """Test different numbers of convolutional blocks."""
    results = {}

    # Configurations
    configs = {
        'Shallow (1 block)': {'num_blocks': 1},
        'Baseline (2 blocks)': {'num_blocks': 2},
        'Deep (3 blocks)': {'num_blocks': 3}
    }

    for name, config in configs.items():
        model_config = {'activation': 'relu', 'num_blocks': config['num_blocks']}
        optimizer_config = {'type': 'adam', 'lr': 0.001}

        result = run_experiment(
            f"A: {name}",
            model_config,
            optimizer_config,
            batch_size=32,
            epochs=15
        )
        results[name] = result

    return results

results_A = experiment_A()


Running Experiment: A: Shallow (1 block)


Epoch 1/15: 100%|██████████| 1563/1563 [00:22<00:00, 70.92it/s]


Epoch 1: Train Loss: 2.0240, Train Acc: 22.01%, Test Loss: 1.7948, Test Acc: 30.08%


Epoch 2/15: 100%|██████████| 1563/1563 [00:22<00:00, 69.08it/s]


Epoch 2: Train Loss: 1.8992, Train Acc: 25.67%, Test Loss: 1.6755, Test Acc: 37.51%


...

Epoch 13: Train Loss: 0.9387, Train Acc: 67.01%, Test Loss: 0.6904, Test Acc: 76.87%


Epoch 14/15: 100%|██████████| 1563/1563 [00:21<00:00, 71.59it/s]


Epoch 14: Train Loss: 0.9113, Train Acc: 68.01%, Test Loss: 0.6737, Test Acc: 77.45%


Epoch 15/15: 100%|██████████| 1563/1563 [00:21<00:00, 71.74it/s]


Epoch 15: Train Loss: 0.8882, Train Acc: 69.39%, Test Loss: 0.6807, Test Acc: 76.77%


Experiment B: Activation Functions

In [7]:
def experiment_B():
    """Test different activation functions."""
    results = {}

    # Configurations
    configs = {
        'ReLU': 'relu',
        'Tanh': 'tanh',
        'Leaky ReLU': 'leaky_relu'
    }

    for name, activation in configs.items():
        model_config = {'activation': activation, 'num_blocks': 2}
        optimizer_config = {'type': 'adam', 'lr': 0.001}

        result = run_experiment(
            f"B: {name}",
            model_config,
            optimizer_config,
            batch_size=32,
            epochs=15
        )
        results[name] = result

    return results

results_B = experiment_B()


Running Experiment: B: ReLU


Epoch 1/15: 100%|██████████| 1563/1563 [00:22<00:00, 70.04it/s]


Epoch 1: Train Loss: 1.8831, Train Acc: 26.78%, Test Loss: 1.5165, Test Acc: 42.76%


Epoch 2/15: 100%|██████████| 1563/1563 [00:22<00:00, 70.07it/s]


Epoch 2: Train Loss: 1.7363, Train Acc: 32.31%, Test Loss: 1.3516, Test Acc: 49.13%


...

Epoch 13: Train Loss: 0.8937, Train Acc: 68.90%, Test Loss: 0.7215, Test Acc: 75.04%


Epoch 14/15: 100%|██████████| 1563/1563 [00:20<00:00, 76.33it/s]


Epoch 14: Train Loss: 0.8855, Train Acc: 69.38%, Test Loss: 0.7007, Test Acc: 76.32%


Epoch 15/15: 100%|██████████| 1563/1563 [00:20<00:00, 74.81it/s]


Epoch 15: Train Loss: 0.8609, Train Acc: 70.08%, Test Loss: 0.6984, Test Acc: 75.68%


Experiment C: Batch Size

In [8]:
def experiment_C():
    """Test different batch sizes."""
    results = {}

    # Configurations
    configs = {
        'Small (16)': 16,
        'Baseline (32)': 32,
        'Large (64)': 64
    }

    for name, batch_size in configs.items():
        model_config = {'activation': 'relu', 'num_blocks': 2}
        optimizer_config = {'type': 'adam', 'lr': 0.001}

        result = run_experiment(
            f"C: {name}",
            model_config,
            optimizer_config,
            batch_size=batch_size,
            epochs=15
        )
        results[name] = result

    return results

results_C = experiment_C()


Running Experiment: C: Small (16)


Epoch 1/15: 100%|██████████| 3125/3125 [00:27<00:00, 114.35it/s]


Epoch 1: Train Loss: 1.8628, Train Acc: 28.76%, Test Loss: 1.4228, Test Acc: 46.95%


Epoch 2/15: 100%|██████████| 3125/3125 [00:27<00:00, 114.78it/s]


Epoch 2: Train Loss: 1.6859, Train Acc: 35.31%, Test Loss: 1.3237, Test Acc: 51.43%


...

Epoch 13: Train Loss: 1.2496, Train Acc: 54.16%, Test Loss: 0.9200, Test Acc: 68.08%


Epoch 14/15: 100%|██████████| 782/782 [00:18<00:00, 41.26it/s]


Epoch 14: Train Loss: 1.2391, Train Acc: 54.33%, Test Loss: 0.9056, Test Acc: 68.95%


Epoch 15/15: 100%|██████████| 782/782 [00:17<00:00, 43.83it/s]


Epoch 15: Train Loss: 1.2259, Train Acc: 54.87%, Test Loss: 0.8933, Test Acc: 69.09%


Experiment D: Optimizers

In [9]:
def experiment_D():
    """Test different optimizers."""
    results = {}

    # Configurations
    configs = {
        'Adam': {'type': 'adam', 'lr': 0.001},
        'SGD (momentum)': {'type': 'sgd', 'lr': 0.01},
        'RMSprop': {'type': 'rmsprop', 'lr': 0.001}
    }

    for name, opt_config in configs.items():
        model_config = {'activation': 'relu', 'num_blocks': 2}

        result = run_experiment(
            f"D: {name}",
            model_config,
            opt_config,
            batch_size=32,
            epochs=15
        )
        results[name] = result

    return results

results_D = experiment_D()


Running Experiment: D: Adam


Epoch 1/15: 100%|██████████| 1563/1563 [00:21<00:00, 71.51it/s]


Epoch 1: Train Loss: 1.8253, Train Acc: 30.82%, Test Loss: 1.4619, Test Acc: 45.97%


Epoch 2/15: 100%|██████████| 1563/1563 [00:21<00:00, 71.68it/s]


Epoch 2: Train Loss: 1.6336, Train Acc: 37.72%, Test Loss: 1.2510, Test Acc: 54.61%


...

Epoch 13: Train Loss: 1.2488, Train Acc: 54.80%, Test Loss: 0.9342, Test Acc: 67.82%


Epoch 14/15: 100%|██████████| 1563/1563 [00:21<00:00, 71.26it/s]


Epoch 14: Train Loss: 1.2130, Train Acc: 56.12%, Test Loss: 0.8948, Test Acc: 69.23%


Epoch 15/15: 100%|██████████| 1563/1563 [00:22<00:00, 70.31it/s]


Epoch 15: Train Loss: 1.1859, Train Acc: 57.58%, Test Loss: 0.9456, Test Acc: 66.07%


Visualization & Analysis Functions

In [10]:
def plot_training_curves(results_dict, experiment_name, metric='train_acc'):
    """
    Plot training accuracy curves for all configurations in an experiment.
    """
    plt.figure(figsize=(10, 6))

    for name, result in results_dict.items():
        history = result['history']
        epochs = range(1, len(history[metric]) + 1)
        plt.plot(epochs, history[metric], label=name, linewidth=2)

    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Test Accuracy (%)', fontsize=12)
    plt.title(f'Experiment {experiment_name}: Test Accuracy vs. Epochs', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'exp_{experiment_name}_curves.png', dpi=300)
    plt.show()

def generate_results_table(all_results):
    """Generate a summary table of all results."""
    data = []
    for experiment, results in all_results.items():
        for config_name, result in results.items():
            data.append({
                'Experiment': experiment,
                'Configuration': config_name,
                'Test Accuracy (%)': f"{result['final_test_acc']:.2f}",
                'Training Time (s)': f"{result['training_time']:.1f}"
            })

    df = pd.DataFrame(data)
    return df

# Generate all results
all_results = {
    'A - Layers': results_A,
    'B - Activations': results_B,
    'C - Batch Size': results_C,
    'D - Optimizers': results_D
}

# Create summary table
summary_df = generate_results_table(all_results)
print("\n" + "="*70)
print("SUMMARY RESULTS TABLE")
print("="*70)
print(summary_df.to_string(index=False))


SUMMARY RESULTS TABLE
     Experiment       Configuration Test Accuracy (%) Training Time (s)
     A - Layers   Shallow (1 block)             50.51             375.9
     A - Layers Baseline (2 blocks)             69.31             387.6
     A - Layers     Deep (3 blocks)             76.77             388.1
B - Activations                ReLU             64.27             368.2
B - Activations                Tanh             65.84             361.5
B - Activations          Leaky ReLU             75.68             361.0
 C - Batch Size          Small (16)             67.69             460.2
 C - Batch Size       Baseline (32)             65.79             359.0
 C - Batch Size          Large (64)             69.09             311.5
 D - Optimizers                Adam             69.86             360.7
 D - Optimizers      SGD (momentum)             71.56             371.0
 D - Optimizers             RMSprop             66.07             366.6
